<a href="https://colab.research.google.com/github/gigimcc4/IBM_for_researchers/blob/main/Tutorial_2_RAG_Under_the_Hood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **IMPORTANT - FIRST - MAKE A COPY OF THIS NOTEBOOK, OR NOTHING WILL BE SAVED! FILE -> SAVE A COPY IN DRIVE/GITHUB**

# Day 3 Tutorial 1 Under the Hood: How RAG Works
## The Conceptual Demo

---
*Created for the Nagoya University 5-Day Faculty Seminar on Teaching with AI*  
**Dr. Jeanne McClure, PhD** | NC State University Data Science Academy / Ars Innovate

---

### 📜 License

© 2026 Dr. Jeanne McClure.

- **NonCommercial** — You may not use the material for commercial purposes.

This material was in part developed with support from the National Science Foundation, DUE-2313644. All opinions, findings, conclusions and recommendations expressed herein are those of the authors and do not necessarily reflect the views of the Foundation.

---

**What this extension is:** A guided look at the machinery behind grounded AI tools
like NotebookLM. You'll see how documents get turned into searchable knowledge,
and why citations sometimes go wrong.

**What this extension is NOT:** A full coding tutorial. The code here is for
**demonstration** - you'll run it, but the goal is understanding the *concepts*,
not memorizing the syntax.

**Time:** ~60 minutes  
**You need:** This Colab notebook (already open!) and curiosity

>  If you'd rather **design a student-facing NotebookLM experience** without
> looking at code, switch to **Extension A - NotebookLM for Your Students.**

---
This material was in part developed with support from the National Science Foundation, DUE-2313644. All opinions, findings, conclusions and recommendations expressed herein are those of the authors and do not necessarily reflect the views of the Foundation.


##  Why Look Under the Hood?

In the main session, you used Gemini embedded in Colab - AI grounded in your **data.**  

And Monday we played in NotebookLM which is AI grounded in your **documents.**

Both are examples of **grounded AI** — but they work differently under the hood.

Gemini in Colab uses **context-window grounding**: your data is directly visible in the notebook environment, and Gemini reads it as part of its input. No searching required.

NotebookLM uses **Retrieval-Augmented Generation (RAG)**: your documents get indexed, and the system *searches* for relevant pieces before generating an answer.

This extension is about that second mechanism — RAG:

```
Traditional AI:  Question  LLM  Answer (from training data - could be anything)

RAG / Grounded:  Question  SEARCH your documents  Feed relevant chunks to LLM  Answer with citations
```

**Why does this matter for architects?**

When you understand the pipeline, you understand:
- Why NotebookLM sometimes gives wrong citations (the search step failed)
- Why document format matters (parsing step)
- Why short questions sometimes get better answers than long ones (embedding step)
- Why the same question can get different answers with different documents (retrieval step)

This knowledge makes you a **better architect** - you can anticipate failure modes
and design better student experiences.


---
##  The RAG Pipeline - Big Picture

Every grounded AI tool (NotebookLM, Copilot with documents, custom RAG systems)
follows this same pipeline:

```

                  THE RAG PIPELINE                       
                                                         
   INGEST         Parse documents into text           
                                                        
    CHUNK          Split text into small pieces        
                                                        
   EMBED          Convert chunks to numbers (vectors)
                                                        
   STORE          Save vectors in a searchable index  
                                                        
   RETRIEVE       Find chunks similar to the question
                                                        
   GENERATE       LLM writes answer using the chunks  
                                                        
   CITE           Attach source references            

```

We'll walk through each step with a small example. **At each step, we'll connect
back to what this means for your students.**


---
##  Step 0: Setup

Run the cell below to install the libraries we need. This takes about 60 seconds.

**You don't need to understand the installation - just run it and wait for **


In [ ]:
# Step 0: Install libraries (run once, ~60 seconds)
!pip install -q sentence-transformers chromadb

print(' Libraries installed!')
print('   sentence-transformers - converts text to numbers')
print('   chromadb - stores and searches those numbers')


---
##  Step 1: INGEST - Parse Documents into Text

**What happens:** The system reads your uploaded files (PDFs, slides, web pages)
and extracts the text.

**Why it can fail:** PDFs are notoriously messy. Tables become scrambled text.
Headers and footers get mixed in. Two-column layouts merge into nonsense.

**What this means for architects:**
> When students get a strange answer from NotebookLM, it might not be the AI's
> fault - the document might have parsed badly. This is a teaching moment about
> **data quality at the source.**

For our demo, we'll create three short 'documents' directly - imagine these
are paragraphs from research papers you uploaded to NotebookLM:


In [ ]:
# Step 1: Our sample "documents"
# In a real system, these would be parsed from PDFs.
# We're using plain text to focus on the concepts.

documents = {
    "Paper A: Chen et al. (2024)": """
Active learning strategies significantly improved student retention in
introductory biology courses. Students who participated in think-pair-share
activities scored 15% higher on final exams compared to traditional lecture
sections. However, the effect was smaller for students who already had strong
study habits. The intervention may be most beneficial for students who lack
structured approaches to learning.
""".strip(),

    "Paper B: Yamamoto & Park (2025)": """
AI-assisted tutoring systems showed mixed results in a large-scale study
across 12 universities. While students reported higher satisfaction and
engagement, exam performance did not significantly differ between AI-tutored
and traditionally-tutored groups. Notably, students using AI tutors spent
30% less time on homework but achieved comparable scores, suggesting
efficiency gains rather than learning gains.
""".strip(),

    "Paper C: Okonkwo (2024)": """
Retrieval practice combined with spaced repetition produced the largest
effect sizes in a meta-analysis of 47 studies. The effect was consistent
across STEM and humanities courses. Importantly, students who used
AI-generated flashcards performed equally well as those using instructor-
created materials, suggesting that the retrieval practice itself - not
the source of the materials - drives the learning benefit.
""".strip(),
}

print(f' Loaded {len(documents)} documents:')
for title, text in documents.items():
    print(f'   {title} ({len(text.split())} words)')


---
##  Step 2: CHUNK - Split Text into Pieces

**What happens:** Each document gets split into smaller pieces called "chunks."
Why? Because AI models have limited attention - giving them a whole textbook
doesn't work. Instead, you give them the most relevant *pieces.*

**The chunking dilemma:**

| Chunks too small | Chunks too big |
|-----------------|----------------|
| Loses context - "15% higher" means nothing without knowing *what* was measured | Dilutes relevance - includes irrelevant text alongside the answer |
| More precise citations | Vaguer citations |
| Like highlighting a single word | Like highlighting the whole page |

**What this means for architects:**
> When NotebookLM cites a long passage, it might be because the chunk was too big.
> When it seems to miss context, the chunk might have been too small.
> You can't control NotebookLM's chunking - but you can control your **document structure.**
> Well-organized documents with clear sections chunk better.

Our documents are short, so each one becomes one chunk. In real systems,
a 20-page paper might become 4080 chunks:


In [ ]:
# Step 2: Create chunks
# For our demo, each document is one chunk.
# Real systems use 200-500 word chunks with overlap.

chunks = []
chunk_sources = []  # Track where each chunk came from

for title, text in documents.items():
    chunks.append(text)
    chunk_sources.append(title)

print(f' Created {len(chunks)} chunks:')
for i, source in enumerate(chunk_sources):
    preview = chunks[i][:80].replace('\n', ' ')
    print(f'   Chunk {i}: [{source}] "{preview}..."')

print(f'\n In NotebookLM, a 10-page PDF might produce 30-50 chunks.')
print(f'   The quality of those chunks depends on your document\'s structure.')


---
##  Step 3: EMBED - The Key Concept

This is the step that makes RAG *work* - and the one most people don't understand.

**The problem:** We need to search documents by *meaning*, not just keywords.  
If a student asks "Does AI help students learn?" we want to find passages about
tutoring effectiveness - even if they never use the word "help."

**The solution: Embeddings**

An embedding model converts text into a list of numbers (a "vector") that
captures its *meaning.* Texts with similar meanings get similar numbers.

```
"cat"   [0.2, 0.8, 0.1, 0.5, ...]    (384 numbers)
"kitten"  [0.2, 0.7, 0.1, 0.6, ...]    (similar numbers! similar meaning!)
"tax"   [0.9, 0.1, 0.7, 0.2, ...]    (different numbers - different meaning)
```

**Analogy:** Imagine each text gets placed on a map. Similar texts are close
together. When you search, you find the nearest texts on the map.

Let's see this in action:


In [ ]:
# Step 3: Create embeddings
from sentence_transformers import SentenceTransformer

# Load a small, fast embedding model
print('Loading embedding model (first time takes ~30 seconds)...')
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f' Model loaded!')
print(f'   This model converts any text into a vector of {model.get_sentence_embedding_dimension()} numbers.')

# Embed our chunks
chunk_vectors = model.encode(chunks)

print(f'\n Embedded {len(chunks)} chunks. Here\'s what one looks like:')
print(f'   Chunk 0 vector (first 10 of {len(chunk_vectors[0])} numbers):')
print(f'   {chunk_vectors[0][:10].round(3).tolist()}')
print(f'\n   These numbers encode the MEANING of the text.')
print(f'   Similar texts  similar numbers  found by search.')


###  Seeing the Meaning Map

Let's visualize where our documents land on the "meaning map."  
We'll also embed some questions and see which documents they're closest to.

**This is exactly what happens when you type a question into NotebookLM** -
your question gets embedded, and the system finds the closest document chunks.


In [ ]:
# Step 3b: Visualize the meaning map
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Some sample questions a student might ask
questions = [
    "Does AI help students learn better?",
    "What teaching methods improve exam scores?",
    "How effective are flashcards for studying?",
]

# Embed the questions too
question_vectors = model.encode(questions)

# Combine all vectors for visualization
all_vectors = np.vstack([chunk_vectors, question_vectors])

# Reduce to 2D for plotting (PCA)
pca = PCA(n_components=2)
coords = pca.fit_transform(all_vectors)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 7))

# Plot document chunks
for i, source in enumerate(chunk_sources):
    short = source.split(':')[0]  # "Paper A", etc.
    ax.scatter(coords[i, 0], coords[i, 1], s=200, marker='s', zorder=5)
    ax.annotate(short, (coords[i, 0], coords[i, 1]),
                fontsize=11, fontweight='bold',
                xytext=(10, 10), textcoords='offset points')

# Plot questions
n = len(chunks)
for i, q in enumerate(questions):
    ax.scatter(coords[n+i, 0], coords[n+i, 1], s=150, marker='*',
               color='red', zorder=5)
    ax.annotate(f'Q: "{q[:35]}..."', (coords[n+i, 0], coords[n+i, 1]),
                fontsize=9, color='red',
                xytext=(10, -15), textcoords='offset points')

    # Draw line to nearest chunk
    dists = [np.linalg.norm(coords[n+i] - coords[j]) for j in range(n)]
    nearest = np.argmin(dists)
    ax.plot([coords[n+i, 0], coords[nearest, 0]],
            [coords[n+i, 1], coords[nearest, 1]],
            'r--', alpha=0.4)

ax.set_title('The Meaning Map: Documents () and Questions ()', fontsize=14)
ax.set_xlabel('Meaning Dimension 1')
ax.set_ylabel('Meaning Dimension 2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(' Questions land near the documents that can answer them.')
print('   This is how NotebookLM decides which passages to cite!')


---
##  Step 4: STORE - Save in a Searchable Index

**What happens:** The vectors get stored in a **vector database** - a special
database optimized for "find the nearest meaning" searches.

**Analogy:** A regular database is like a filing cabinet organized alphabetically.
A vector database is like a room where similar documents are placed physically
close together. To search, you walk to the area that matches your question.

We'll use ChromaDB - the same kind of technology that powers NotebookLM's search:


In [ ]:
# Step 4: Store in a vector database
import chromadb

# Create a small in-memory database
client = chromadb.Client()
collection = client.create_collection(
    name="demo_papers",
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity
)

# Add our chunks with their source info
collection.add(
    documents=chunks,
    metadatas=[{"source": s} for s in chunk_sources],
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=chunk_vectors.tolist(),
)

print(f' Stored {collection.count()} chunks in the vector database.')
print(f'\n   Think of this as NotebookLM\'s internal index.')
print(f'   When you upload documents, this is what gets built behind the scenes.')


---
##  Step 5: RETRIEVE - Find the Relevant Chunks

**What happens:** When you ask a question, it gets embedded into a vector,
then the database finds the chunks with the most similar vectors.

**This is where the magic happens - and where things can go wrong:**

| Retrieval succeeds when... | Retrieval fails when... |
|---------------------------|------------------------|
| Question and answer use similar concepts | Question uses different vocabulary than the documents |
| The relevant passage is a self-contained chunk | The answer spans multiple chunks that got separated |
| One clear best match exists | Multiple chunks are equally relevant |

Let's try some searches and see what comes back:


In [ ]:
# Step 5: Retrieve - search by meaning

def search(question, n_results=2):
    """Search our vector database and show results."""
    # Embed the question
    q_vector = model.encode([question]).tolist()

    # Search
    results = collection.query(
        query_embeddings=q_vector,
        n_results=n_results,
    )

    print(f' Question: "{question}"')
    print(f'   Top {n_results} results:\n')

    for i in range(n_results):
        source = results['metadatas'][0][i]['source']
        distance = results['distances'][0][i]
        text = results['documents'][0][i][:120].replace('\n', ' ')
        similarity = round(1 - distance, 3)  # Convert distance to similarity

        print(f'   [{i+1}] {source}')
        print(f'       Similarity: {similarity} (1.0 = perfect match)')
        print(f'       "{text}..."')
        print()

    return results

# Try it!
print('=' * 70)
_ = search("Does AI help students learn better?")
print('=' * 70)
_ = search("What is the most effective study technique?")
print('=' * 70)
_ = search("How do active learning strategies affect biology students?")


###  Try Your Own Question

Change the question below and run the cell to see what the retrieval engine finds.

**Try these experiments:**
1. Ask something the documents cover  you should get a relevant result
2. Ask something the documents DON'T cover (e.g., "What is quantum physics?")  see what happens
3. Ask a broad question vs. a specific question  notice the similarity scores change


In [ ]:
#  EXPERIMENT: Change this question and run the cell!

my_question = "What role does AI play in education?"  #  Change this!

results = search(my_question)

# Check: is the top result actually relevant?
top_similarity = round(1 - results['distances'][0][0], 3)
if top_similarity > 0.5:
    print(' High similarity - the retrieval engine found a relevant passage.')
elif top_similarity > 0.3:
    print(' Medium similarity - the result might be relevant, but the match is weak.')
    print('   This is where NotebookLM might give a vague or partially wrong answer.')
else:
    print(' Low similarity - no good match found.')
    print('   A well-designed system would say "I don\'t know" here.')
    print('   A poorly designed system might hallucinate an answer anyway.')


---
##  Step 6: GENERATE - The LLM Writes an Answer

**What happens:** The retrieved chunks get packaged into a prompt for the LLM:

```
SYSTEM: Answer the user's question using ONLY the provided context.
        Cite your sources using [1], [2], etc.
        If the context doesn't contain the answer, say so.

CONTEXT:
[1] Chen et al. (2024): Active learning strategies significantly improved
    student retention in introductory biology courses...

[2] Yamamoto & Park (2025): AI-assisted tutoring systems showed mixed
    results in a large-scale study across 12 universities...

USER QUESTION: Does AI help students learn better?
```

**The LLM then writes a response grounded in those chunks.**

This is exactly what NotebookLM does - your question + the retrieved passages
become the LLM's input. The LLM can only work with what the retrieval step gave it.

### Where Things Go Wrong at This Step

| Failure Mode | What Happened | Extension A Checklist Criterion |
|-------------|---------------|--------------------------------|
| **Confident misquote** | LLM rephrases but changes the meaning ("may improve"  "improves") | Criterion 3: Fidelity |
| **Cherry-picking** | LLM uses [1] but ignores [2] which contradicts it | Criterion 4: Synthesis |
| **Hallucination beyond context** | LLM adds facts from its training data, not the documents | Criterion 1: Accuracy |
| **Missing the best passage** | Retrieval found a mediocre match; the best passage wasn't retrieved | Criterion 5: Gaps |

>  **The architect's insight:** Most "AI errors" in NotebookLM come from Step 5
> (bad retrieval) or Step 6 (bad generation), not from the documents themselves.
> But you can reduce Step 5 errors by uploading well-structured documents.


###  See the Prompt That Gets Built

Let's see exactly what prompt would be sent to the LLM after retrieval.
This is the "behind the curtain" moment - **this is what NotebookLM builds
every time you ask a question:**


In [ ]:
# Step 6: Build the RAG prompt (demonstration)

def build_rag_prompt(question, n_results=2):
    """Show the exact prompt that would be sent to an LLM."""
    q_vector = model.encode([question]).tolist()
    results = collection.query(query_embeddings=q_vector, n_results=n_results)

    # Build the prompt
    prompt = """SYSTEM: You are a research assistant. Answer the user's question
using ONLY the provided context. Cite your sources using [1], [2], etc.
If the context doesn't contain enough information, say so honestly.

CONTEXT:\n"""

    for i in range(n_results):
        source = results['metadatas'][0][i]['source']
        text = results['documents'][0][i]
        prompt += f"\n[{i+1}] {source}:\n{text}\n"

    prompt += f"\nUSER QUESTION: {question}\n"

    return prompt

# Build and display
question = "Does AI help students learn better?"
prompt = build_rag_prompt(question)

print('=' * 70)
print('THE PROMPT THAT NOTEBOOKLM BUILDS (conceptually):')
print('=' * 70)
print(prompt)
print('=' * 70)
print()
print(' Everything the LLM knows comes from CONTEXT above.')
print('   If a passage is missing, the LLM cannot use it.')
print('   If a passage is irrelevant, the LLM might get confused.')
print('   The quality of RETRIEVAL (Step 5) determines the quality of GENERATION.')


---
##  Connecting the Pipeline to the Student Audit

Now you can see **why each criterion in the Citation Audit Checklist exists:**

| Audit Criterion | Pipeline Step Where It Fails | What Students Should Ask |
|----------------|------------------------------|-------------------------|
| **1. Accuracy** | Step 6 (Generation) | Did the AI read the cited passage correctly? |
| **2. Completeness** | Step 5 (Retrieval) | Did the AI find the *best* passage, or just the first match? |
| **3. Fidelity** | Step 6 (Generation) | Did the AI preserve qualifiers like "may" and "suggests"? |
| **4. Synthesis** | Step 6 (Generation) | When combining sources, did AI treat them fairly? |
| **5. Gaps** | Steps 15 (Ingest through Retrieve) | What's missing? Was it in the documents but not retrieved? Or not in the documents at all? |
| **6. Your reading** | *Human judgment* | What do YOU think after reading the cited passage? |

### The Architect's Advantage

**If you only use NotebookLM (Extension A),** you know *that* things go wrong.  
**If you also understand RAG (Extension B),** you know *why* things go wrong -
and you can design better student experiences:

- Upload well-structured documents  better chunking (Step 2)
- Use clear section headings  better retrieval (Step 5)
- Include documents that disagree  forces honest synthesis (Step 6)
- Exclude some documents deliberately  teaches students about knowledge gaps


---
##  Experiment: When Retrieval Fails

Let's deliberately trigger each failure mode so you can recognize them
when your students encounter them:


In [ ]:
# Experiment: Retrieval failure modes

print('FAILURE MODE 1: Question outside the knowledge base')
print('=' * 60)
_ = search("What causes climate change?")
print(' The system returns SOMETHING - but the similarity is low.')
print('   A good system says "I don\'t know." A bad one uses these weak matches.\n')

print('FAILURE MODE 2: Vocabulary mismatch')
print('=' * 60)
_ = search("pedagogical interventions in tertiary STEM education")
print(' The documents discuss this topic, but use different words.')
print('   Embedding models are *usually* good at this, but not always.\n')

print('FAILURE MODE 3: The question is too broad')
print('=' * 60)
_ = search("Tell me about education")
print(' Everything matches weakly, nothing matches strongly.')
print('   This is why specific questions get better answers than vague ones.')


---
##  NotebookLM vs. Custom RAG - When Does Each Make Sense?

Now that you've seen the pipeline, you can make an informed choice:

| | NotebookLM (Extension A) | Custom RAG (What we built here) |
|---|---|---|
| **Setup time** | 2 minutes | 2 hours to 2 weeks |
| **Control over pipeline** | None - Google controls everything | Total - you choose every component |
| **Best for students** | Yes - zero friction, immediate use | No - too much technical overhead |
| **Best for understanding** | Partial - you see inputs and outputs | Yes - you see every step |
| **Best for research** | Quick literature exploration | Custom pipelines for specific needs |
| **Citation quality** | Good (Google's system) | Depends on your chunking and retrieval |
| **Cost** | Free (with Google account) | Varies (embedding models, compute) |

### The Architect's Recommendation

> **For student-facing activities:** Use NotebookLM. The audit experience is
> identical, and students can focus on *evaluating* rather than *building.*
>
> **For your own understanding:** This Extension B gives you the mental model
> to debug student confusion when NotebookLM behaves unexpectedly.
>
> **For research workflows:** Custom RAG gives you control - specific embedding
> models for your domain, custom chunking for your document types, integration
> with your existing tools.


---

Now that you understand the pipeline, here's what to prepare:

**If you chose the  Student Track:**
- Think about how your document structure affects chunking
- Consider: should you pre-process your PDFs for better parsing?
- Bring documents with clear headings and sections

**If you chose the  Research Track:**
- Consider whether NotebookLM's default pipeline is sufficient, or if you
  need custom embedding models for your domain's vocabulary
- Identify: what types of questions will you ask your literature?

**If you chose the  Department Track:**
- Think about document organization - can you structure department materials
  to chunk well in a grounded AI system?
- Consider: what knowledge is currently scattered across documents that
  would benefit from unified search?


---
##  Key Vocabulary

| Term | Plain English | Analogy |
|------|--------------|--------|
| **RAG** | Retrieval-Augmented Generation - AI that searches before answering | Looking up your notes before answering a question |
| **Embedding** | Converting text to numbers that capture meaning | Placing books on a map by topic |
| **Vector** | The list of numbers representing a text's meaning | A coordinate on the meaning map |
| **Chunk** | A piece of a document stored for retrieval | A highlighted passage in a textbook |
| **Vector database** | A searchable collection of embeddings | A library organized by topic, not alphabet |
| **Cosine similarity** | How similar two vectors are (0 = unrelated, 1 = identical meaning) | How close two books are on the map |
| **Grounded AI** | AI that answers from specific sources, not general training | A student answering from the assigned readings, not Wikipedia |


---
##  Extension RAG Wrap-Up

**What you learned today:**
-  The 7-step RAG pipeline (Ingest  Chunk  Embed  Store  Retrieve  Generate  Cite)
-  Why embeddings are the key technology behind grounded AI
-  How retrieval quality determines answer quality
-  Why document structure matters for AI performance
-  How each audit criterion maps to a pipeline step

**The key insight:**

> Grounded AI doesn't "read" your documents the way humans do.  
> It converts them to numbers, searches by similarity, and generates from chunks.  
> Every failure in the pipeline is a teaching moment for students.

---


---
* 2026 Dr. Jeanne McClure  
